# Etna Cause–Trigger Analysis

This notebook runs the Cause–Trigger algorithm on the Etna dataset using three causal-discovery backends:

1. **HMML** — baseline.
2. **PCMCI** — lagged causal-discovery sensitivity backend.
3. **PCMCI+** — lagged + contemporaneous causal-discovery backend; only lagged `tau >= 1` parents are fed into the Cause–Trigger F-test, while contemporaneous `tau = 0` links are stored as diagnostics.

The goal is to keep the analysis tidy:

- define one dataset and one effect variable,
- run all backends with the same lag/alpha/window settings,
- compare split, B1/B2, trigger candidates, accepted triggers, diagnostics, and PCMCI+ contemporaneous links,
- then run a controlled sensitivity grid.

## 1. Imports and paths
This section loads the local modules from `src/` and sets data paths.  

In [ ]:
from pathlib import Path
import sys
import importlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data_etna"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import cause_trigger
import parameter_extraction

importlib.reload(cause_trigger)
importlib.reload(parameter_extraction)

from cause_trigger import (
    CauseTriggerConfig,
    run_cause_trigger,
    find_increase_split,
    diagnostics_to_dataframe,
)

from parameter_extraction import find_parameters

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

## 2. Load and prepare Etna dataset

The Cause–Trigger module expects a fully numeric, standardized/scaled dataframe with a time index and no missing values.  
The `station` column is metadata and is removed.

In [ ]:
etna_path = DATA_DIR / "FINAL_ME01_scaled.csv"

etna = pd.read_csv(etna_path, parse_dates=["time"])

print("Raw loaded shape:", etna.shape)
display(etna.head())
display(etna.isna().mean().sort_values())

X_all = (
    etna
    .drop(columns=["station"], errors="ignore")
    .set_index("time")
    .sort_index()
)

X_all = X_all.select_dtypes(include=[np.number])

print("Model dataframe shape:", X_all.shape)
print("Start:", X_all.index.min())
print("End:", X_all.index.max())
display(X_all.head())
display(X_all.isna().sum())

if X_all.isna().any().any():
    raise ValueError("X_all contains NaNs. Fix before running Cause–Trigger.")

## 3. Analysis configuration

For the first controlled comparison, all three backends use the same:

- effect variable,
- lag length,
- alpha,
- split-length constraints,
- Gaussian distribution,
- `beta_is_ones=True` for baseline behavior.

For Etna, `effect_seismic_scaled` is treated as the effect/response variable.  

In [ ]:
EFFECT = "effect_seismic_scaled"
ETNA_EVENT_TIME = pd.Timestamp("2008-05-12 06:28:00", tz="UTC")

ALPHA = 0.05
MANUAL_LAG = 3
MAX_LAGS = 6

MIN_I1_LENGTH = 48
MIN_I2_LENGTH = 48

SELECTED_DISTRIBUTION = "gaussian"

X_FULL = X_all.copy()
df_model = X_FULL.copy()

print("Effect variable:", EFFECT)
print("Manual lag:", MANUAL_LAG)
print("Distribution:", SELECTED_DISTRIBUTION)
print("Known event time:", ETNA_EVENT_TIME)
print("Dataset:", X_FULL.index.min(), "to", X_FULL.index.max())
print("Shape:", X_FULL.shape)

## 4. Split diagnostics before running causal discovery

This checks whether the split condition behaves sensibly for the chosen effect.  
The split is selected by maximizing:

`|E(effect)|_I2 - |E(effect)|_I1`

subject to the minimum interval lengths.

In [ ]:
def split_diagnostics(
    X,
    effect,
    event_time=None,
    min_I1_length=48,
    min_I2_length=48,
):
    split_idx = find_increase_split(
        X[effect],
        min_I1_length=min_I1_length,
        min_I2_length=min_I2_length,
    )

    if split_idx is None:
        return {
            "effect": effect,
            "split_index": None,
            "split_time": None,
            "I1_length": None,
            "I2_length": None,
            "abs_mean_I1": None,
            "abs_mean_I2": None,
            "abs_mean_difference": None,
            "distance_to_event": None,
        }

    I1 = X.iloc[:split_idx]
    I2 = X.iloc[split_idx:]
    split_time = X.index[split_idx]

    out = {
        "effect": effect,
        "split_index": split_idx,
        "split_time": split_time,
        "I1_length": len(I1),
        "I2_length": len(I2),
        "abs_mean_I1": abs(I1[effect].mean()),
        "abs_mean_I2": abs(I2[effect].mean()),
        "abs_mean_difference": abs(I2[effect].mean()) - abs(I1[effect].mean()),
        "distance_to_event": None,
    }

    if event_time is not None:
        out["distance_to_event"] = split_time - event_time

    return out


split_table = pd.DataFrame([
    split_diagnostics(
        X_FULL,
        effect=EFFECT,
        event_time=ETNA_EVENT_TIME,
        min_I1_length=MIN_I1_LENGTH,
        min_I2_length=MIN_I2_LENGTH,
    )
])

display(split_table)

## 5. Shared result helpers

These functions keep the backend sections tidy and ensure every run is summarized in the same way.

In [ ]:
def summarize_result(result, title=None):
    if title:
        print("\n" + "=" * 100)
        print(title)
        print("=" * 100)

    print("Backend:", result.get("backend"))
    print("Configured lags:", result.get("configured_lags"))
    print("Configured distribution:", result.get("configured_distribution"))
    print("Parameter source:", result.get("parameter_source"))
    print("Split index:", result.get("split_index"))
    print("Split timestamp:", result.get("split_timestamp"))
    print("I1 length:", result.get("I1_length"))
    print("I2 length:", result.get("I2_length"))
    print("Effect abs mean difference:", result.get("target_abs_mean_difference"))
    print("B1:", result.get("B_1"))
    print("B2:", result.get("B_2"))
    print("Autoregressive effect parent in B2:", result.get("autoregressive_parent_in_B2"))
    print("Trigger candidates:", result.get("T_candidates"))
    print("Accepted triggers:", result.get("T"))
    print("Causes:", result.get("C"))
    print("Cause-trigger pairs:", result.get("pairs"))

    scores = result.get("causal_scores", {})

    if "_hmml_metadata" in scores:
        print("HMML metadata:")
        display(scores["_hmml_metadata"])

    if "_pcmci_metadata" in scores:
        print("PCMCI/PCMCI+ metadata:")
        display(scores["_pcmci_metadata"])

    contemp = result.get("contemporaneous_links", {})
    if contemp:
        print("PCMCI+ contemporaneous tau=0 links to effect, stored as diagnostics only:")
        display(pd.DataFrame.from_dict(contemp, orient="index"))

    if result.get("selected_cause_shift_scores"):
        print("Cause-selection shift scores by trigger:")
        display(result["selected_cause_shift_scores"])

    diag = diagnostics_to_dataframe(result)

    if len(diag) > 0:
        sort_cols = [c for c in ["accepted", "p_value", "rss_reduction_ratio"] if c in diag.columns]
        if sort_cols:
            ascending = []
            for c in sort_cols:
                ascending.append(False if c in ["accepted", "rss_reduction_ratio"] else True)
            diag = diag.sort_values(sort_cols, ascending=ascending)

        display(diag)

    return diag


def compact_result_row(name, effect, result):
    return {
        "run": name,
        "effect": effect,
        "backend": result.get("backend"),
        "parameter_source": result.get("parameter_source"),
        "configured_lags": result.get("configured_lags"),
        "configured_distribution": result.get("configured_distribution"),
        "split_index": result.get("split_index"),
        "split_timestamp": result.get("split_timestamp"),
        "I1_length": result.get("I1_length"),
        "I2_length": result.get("I2_length"),
        "B1": result.get("B_1"),
        "B2": result.get("B_2"),
        "autoregressive_effect_parent_in_B2": result.get("autoregressive_parent_in_B2"),
        "trigger_candidates": result.get("T_candidates"),
        "accepted_triggers": result.get("T"),
        "causes": result.get("C"),
        "pairs": result.get("pairs"),
        "n_contemporaneous_links": len(result.get("contemporaneous_links", {}) or {}),
    }


def add_run_label(diag, run_name, effect, backend, parameter_source):
    if diag is None or len(diag) == 0:
        return pd.DataFrame()

    out = diag.copy()
    out["run"] = run_name
    out["effect"] = effect
    out["backend"] = backend
    out["parameter_source"] = parameter_source
    return out


def plot_effect_with_split(df, effect, result, event_time=None, title_suffix=""):
    plt.figure(figsize=(16, 4))
    plt.plot(df.index, df[effect], label=effect, linewidth=0.8)

    split_timestamp = result.get("split_timestamp")
    if split_timestamp is not None:
        plt.axvline(split_timestamp, linestyle="--", label="Detected split")

    if event_time is not None:
        event_time = pd.Timestamp(event_time)
        plt.axvline(event_time, linestyle=":", label="Known event time")

    plt.title(f"Effect {effect}: detected split vs known event {title_suffix}")
    plt.xlabel("Time")
    plt.ylabel("Scaled value")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Shared backend runner

This function creates a `CauseTriggerConfig`, runs the algorithm, and returns the result and diagnostics.  
The backend-specific settings are controlled through `causal_backend`.

In [ ]:
def make_config(
    backend,
    effect=EFFECT,
    lag=MANUAL_LAG,
    distribution=SELECTED_DISTRIBUTION,
    parameter_source=None,
    cond_ind_test="parcorr",
):
    if parameter_source is None:
        parameter_source = f"manual_lag{lag}_{distribution}_full_dataset"

    return CauseTriggerConfig(
        y_t=effect,
        lags=lag,
        distribution=distribution,
        causal_backend=backend,
        alpha=ALPHA,
        min_I1_length=MIN_I1_LENGTH,
        min_I2_length=MIN_I2_LENGTH,
        beta_is_ones=True,
        parameter_source=parameter_source,

        # PCMCI / PCMCI+ settings
        pcmci_pc_alpha=0.05,
        pcmci_alpha_level=0.05,
        pcmci_fdr_method="fdr_bh",
        pcmci_cond_ind_test=cond_ind_test,
        pcmci_verbosity=0,

        # PCMCI+ settings
        pcmci_contemp_collider_rule="majority",
        pcmci_conflict_resolution=True,
        pcmci_keep_raw_results=False,
    )


def run_backend(
    backend,
    X=X_FULL,
    effect=EFFECT,
    lag=MANUAL_LAG,
    distribution=SELECTED_DISTRIBUTION,
    cond_ind_test="parcorr",
):
    config = make_config(
        backend=backend,
        effect=effect,
        lag=lag,
        distribution=distribution,
        parameter_source=f"{backend}_manual_lag{lag}_{distribution}_full_dataset",
        cond_ind_test=cond_ind_test,
    )

    result = run_cause_trigger(X, config)
    diag = summarize_result(result, title=f"{backend.upper()} result")
    return result, diag

## 7. Run Backends + compare

- HMML
- PCMCI lagged causal-discovery sensitivity
- PCMCI+ causal-discovery sensitivity

PCMCI is used here only for lagged parent discovery.  

PCMCI+ can identify both lagged and contemporaneous links.  
For the Cause–Trigger algorithm, only lagged `tau >= 1` links are used in `B2`.  
Contemporaneous `tau = 0` links are stored separately in `result['contemporaneous_links']`.

The table compares the key algorithm outputs across HMML, PCMCI, and PCMCI+.

In [ ]:
result_hmml, diag_hmml = run_backend(
    backend="hmml",
    X=X_FULL,
    effect=EFFECT,
    lag=MANUAL_LAG,
    distribution=SELECTED_DISTRIBUTION,
)

In [ ]:
result_pcmci, diag_pcmci = run_backend(
    backend="pcmci",
    X=X_FULL,
    effect=EFFECT,
    lag=MANUAL_LAG,
    distribution=SELECTED_DISTRIBUTION,
    cond_ind_test="parcorr",
)

In [ ]:
result_pcmci_plus, diag_pcmci_plus = run_backend(
    backend="pcmci_plus",
    X=X_FULL,
    effect=EFFECT,
    lag=MANUAL_LAG,
    distribution=SELECTED_DISTRIBUTION,
    cond_ind_test="parcorr",
)

print("PCMCI+ contemporaneous links:")
display(pd.DataFrame.from_dict(result_pcmci_plus.get("contemporaneous_links", {}), orient="index"))

In [ ]:
results_by_backend = {
    "hmml": result_hmml,
    "pcmci": result_pcmci,
    "pcmci_plus": result_pcmci_plus,
}

comparison = pd.DataFrame([
    compact_result_row("hmml_manual", EFFECT, result_hmml),
    compact_result_row("pcmci_manual", EFFECT, result_pcmci),
    compact_result_row("pcmci_plus_manual", EFFECT, result_pcmci_plus),
])

display(comparison)

## 8. Combined moderation diagnostics

This combines the F-test diagnostics for all trigger candidates tested by all three backends.

In [ ]:
all_diag = pd.concat(
    [
        add_run_label(
            diag_hmml,
            "hmml_manual",
            EFFECT,
            "hmml",
            f"manual_lag{MANUAL_LAG}_gaussian_full_dataset",
        ),
        add_run_label(
            diag_pcmci,
            "pcmci_manual",
            EFFECT,
            "pcmci",
            f"manual_lag{MANUAL_LAG}_gaussian_full_dataset",
        ),
        add_run_label(
            diag_pcmci_plus,
            "pcmci_plus_manual",
            EFFECT,
            "pcmci_plus",
            f"manual_lag{MANUAL_LAG}_gaussian_full_dataset",
        ),
    ],
    ignore_index=True,
)

if len(all_diag) > 0:
    sort_columns = [c for c in ["accepted", "p_value", "rss_reduction_ratio"] if c in all_diag.columns]
    ascending = []
    for c in sort_columns:
        ascending.append(False if c in ["accepted", "rss_reduction_ratio"] else True)

    display(all_diag.sort_values(sort_columns, ascending=ascending))
else:
    display(all_diag)

## 9. Split visualization

The split is determined by the effect variable and should be identical across backends if the same input data and split settings are used.

In [ ]:
plot_effect_with_split(
    df=X_FULL,
    effect=EFFECT,
    result=result_hmml,
    event_time=ETNA_EVENT_TIME,
    title_suffix="— HMML baseline",
)

## 10. Optional automatic lag selection

This section uses VAR/AIC to select a lag.  
Because the scaled variables can be negative, the distribution is forced to Gaussian.

In [ ]:
_, selected_lag = find_parameters(
    X=X_FULL,
    target_series=X_FULL[EFFECT],
    max_lags=MAX_LAGS,
    criterion="aic",
    fallback_lag=1,
    fallback_distribution="gaussian",
)

selected_distribution = "gaussian"

print("Effect:", EFFECT)
print("Selected lag:", selected_lag)
print("Forced distribution:", selected_distribution)

## 11. Automatic-lag comparison across three backends

This repeats the three-backend comparison using the selected lag.

In [ ]:
auto_results = {}
auto_diags = {}

for backend in ["hmml", "pcmci", "pcmci_plus"]:
    try:
        config = make_config(
            backend=backend,
            effect=EFFECT,
            lag=selected_lag,
            distribution=selected_distribution,
            parameter_source="VAR_AIC_lag_gaussian_scaled_full_dataset",
            cond_ind_test="parcorr",
        )

        result = run_cause_trigger(X_FULL, config)
        diag = summarize_result(result, title=f"{backend.upper()} auto-lag result")

        auto_results[backend] = result
        auto_diags[backend] = diag

    except Exception as e:
        print(f"{backend} auto-lag run failed:", e)

auto_comparison = pd.DataFrame([
    compact_result_row(f"{backend}_auto_lag", EFFECT, result)
    for backend, result in auto_results.items()
])

display(auto_comparison)

## 12. Sensitivity grid across backend and lag

This section tests stability across:

- backend: HMML, PCMCI, PCMCI+
- lag: 1 to `MAX_LAGS`
- distribution: Gaussian

For Etna, we start with `parcorr`.

In [ ]:
def run_sensitivity_grid(
    df_model,
    effect,
    backends=("hmml", "pcmci", "pcmci_plus"),
    lags=(1, 2, 3),
    distributions=("gaussian",),
    min_I1_length=48,
    min_I2_length=48,
    cond_ind_test="parcorr",
):
    rows = []

    for backend in backends:
        for lag in lags:
            for distribution in distributions:
                config = make_config(
                    backend=backend,
                    effect=effect,
                    lag=lag,
                    distribution=distribution,
                    parameter_source="sensitivity_grid",
                    cond_ind_test=cond_ind_test,
                )

                # Explicitly set interval lengths in case the function arguments differ
                config.min_I1_length = min_I1_length
                config.min_I2_length = min_I2_length

                try:
                    result = run_cause_trigger(df_model, config)
                    diagnostics = diagnostics_to_dataframe(result)

                    rows.append({
                        "effect": effect,
                        "backend": backend,
                        "lag": lag,
                        "distribution": distribution,
                        "cond_ind_test": cond_ind_test if backend in {"pcmci", "pcmci_plus"} else None,
                        "split_timestamp": result.get("split_timestamp"),
                        "I1_length": result.get("I1_length"),
                        "I2_length": result.get("I2_length"),
                        "B_1": result.get("B_1"),
                        "B_2": result.get("B_2"),
                        "autoregressive_effect_parent_in_B2": result.get("autoregressive_parent_in_B2"),
                        "trigger_candidates": result.get("T_candidates"),
                        "accepted_triggers": result.get("T"),
                        "causes": result.get("C"),
                        "pairs": result.get("pairs"),
                        "n_contemporaneous_links": len(result.get("contemporaneous_links", {}) or {}),
                        "n_diagnostics": len(diagnostics),
                        "min_p_value": diagnostics["p_value"].min()
                            if "p_value" in diagnostics.columns and len(diagnostics) > 0
                            else None,
                        "max_rss_reduction_ratio": diagnostics["rss_reduction_ratio"].max()
                            if "rss_reduction_ratio" in diagnostics.columns and len(diagnostics) > 0
                            else None,
                    })

                except Exception as e:
                    rows.append({
                        "effect": effect,
                        "backend": backend,
                        "lag": lag,
                        "distribution": distribution,
                        "cond_ind_test": cond_ind_test if backend in {"pcmci", "pcmci_plus"} else None,
                        "error": str(e),
                    })

    return pd.DataFrame(rows)


sensitivity_df = run_sensitivity_grid(
    df_model=X_FULL,
    effect=EFFECT,
    backends=("hmml", "pcmci", "pcmci_plus"),
    lags=tuple(range(1, MAX_LAGS + 1)),
    distributions=("gaussian",),
    min_I1_length=MIN_I1_LENGTH,
    min_I2_length=MIN_I2_LENGTH,
    cond_ind_test="parcorr",
)

display(sensitivity_df)

## 13. Accepted-trigger stability

This table shows only configurations that produced accepted triggers.  

In [ ]:
if "accepted_triggers" in sensitivity_df.columns:
    accepted_sensitivity = sensitivity_df[
        sensitivity_df["accepted_triggers"].apply(lambda x: isinstance(x, list) and len(x) > 0)
    ].copy()
else:
    accepted_sensitivity = pd.DataFrame()

print("Sensitivity runs with accepted triggers:")
display(accepted_sensitivity)

if len(accepted_sensitivity) == 0:
    print("No accepted triggers under the tested configurations.")
else:
    print("Accepted trigger configurations found. Check whether they are stable across lag/backend.")

## 14. PCMCI/PCMCI+ conditional-test sensitivity

In [ ]:
RUN_ROBUST_PAR_CORR = False

if RUN_ROBUST_PAR_CORR:
    sensitivity_robust_df = run_sensitivity_grid(
        df_model=X_FULL,
        effect=EFFECT,
        backends=("pcmci", "pcmci_plus"),
        lags=tuple(range(1, MAX_LAGS + 1)),
        distributions=("gaussian",),
        min_I1_length=MIN_I1_LENGTH,
        min_I2_length=MIN_I2_LENGTH,
        cond_ind_test="robust_parcorr",
    )

    display(sensitivity_robust_df)
else:
    print("Set RUN_ROBUST_PAR_CORR = True to run robust_parcorr sensitivity.")